In [ ]:
'''
pip install --no-cache-dir \
    ipywidgets==8.1.8 \
    lightning==2.6.0 \
    lightning-utilities==0.15.2 \
    nbformat==5.10.4 \
    neuralforecast==3.1.2 \
    numpy==2.2.6 \
    openpyxl==3.1.5 \
    optuna==4.6.0 \
    plotly==6.5.0 \
    pytorch-lightning==2.6.0 \
    scikit-learn==1.7.2 \
    statsforecast==2.0.3 \
    torch==2.9.1 \
    torchmetrics==1.8.2
'''

# Funções auxiliares

In [1]:
import os
import random
import numpy as np
import torch

# =============================================================================
# 1. CONFIGURAÇÃO CRÍTICA DE REPRODUTIBILIDADE (PRÉ-IMPORTAÇÃO)
# =============================================================================
# Estas variáveis DEVEM ser definidas ANTES de importar lightning ou neuralforecast
# para garantir que o CUDA inicialize em modo determinístico.

global_seed = 42 # Seed global, os pontos onde se usa seed devem usar esta variável.

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' # Configura o tamanho do buffer de memória para o CUDA. Necessario para evitar erro no use_deterministic_algorithms(True).
os.environ["PL_DETERMINISTIC"] = "1" # Faz a biblioteca Lightning operar em modo determinístico.
os.environ['PYTHONHASHSEED'] = str(global_seed) # Fixa a semente de hash do Python para fixar a ordem de iteração em dicionários (dict) e conjuntos (set).

# A API V8 tentar "adivinhar" qual o melhor algoritmo para sua placa de vídeo (algoritmos para acelerar redes neurais).
# Às vezes, essa escolha pode mudar entre uma execução e outra. Alguns desses algoritmos apresentaram comportamentos ligeiramente não-determinísticos (pequenas variações numéricas).
os.environ['TORCH_CUDNN_V8_API_ENABLED'] = '0' 

print("🔒 Variáveis de ambiente de determinismo configuradas pré-importação.")

# =============================================================================
# 2. IMPORTS
# =============================================================================
from neuralforecast.losses.pytorch import HuberLoss
from sklearn.metrics import root_mean_squared_error
from torch.optim.lr_scheduler import StepLR
from neuralforecast import NeuralForecast
from IPython.display import clear_output
from torch.utils.data import DataLoader
from neuralforecast.models import LSTM
from typing import Dict, Any, Optional
from optuna.samplers import TPESampler
from datetime import datetime
from torch.optim import AdamW
from pathlib import Path
import pandas as pd
import optuna
import shutil
import json
import time
import yaml
import sys
import gc # Importante para limpeza de memória

from datetime import datetime, timedelta
# Importação da ferramenta oficial de reprodutibilidade do Lightning
from lightning.pytorch import seed_everything

# =============================================================================
# 3. FUNÇÃO DE REPRODUTIBILIDADE SEGURA
# =============================================================================
def set_reproducibility(seed=42):
    print(f"🔒 Aplicando configurações de reprodutibilidade total (Seed: {seed})...")
    
    # 1. Sementes básicas
    #random.seed(seed) # Seed para a biblioteca random do Python (seed_everything já faz isso internamente).
    #np.random.seed(seed) # Seed para a biblioteca Numpy (seed_everything já faz isso internamente).
    #torch.manual_seed(seed) # Controla a geração de números aleatórios do PyTorch na CPU (inicialização de pesos, tensores aleatórios). (seed_everything já faz isso internamente).
    
    # 2. Configurações de precisão de Matriz 
    # Placas de vídeo modernas usam uma otimização chamada TensorFloat-32 (TF32),
    # que sacrifica um pouco de precisão para multiplicar matrizes muito mais rápido.
    # Essa "imprecisão" gera ruído numérico que varia levemente a cada execução.
    # Ao definir como 'highest' o PyTorch usa a precisão máxima de ponto flutuante (FP32), mesmo que seja mais lento.
    # Isso reduz erros de arredondamento que se acumulam durante o treino.
    torch.set_float32_matmul_precision('highest') 
    
    # 3. A "Bala de Prata" do Lightning
    # Ele garante que o DataLoader e seus workers usem seeds determinísticas.
    # Define as sementes do Python, do NumPy, do PyTorch (CPU) e do PyTorch (GPU).
    # Com um DataLoader com mais de 1 num_workers (procesos em paralelo), o Python cria sub-processos.
    # Sem p workers=True, cada sub-processo poderia iniciar com uma semente aleatória diferente.
    # Com o workers=True ele garante que cada processo de carregamento de dados receba uma semente derivada da semente mestre de forma determinística.
    seed_everything(seed, workers=True)
    
    # 4. Força algoritmos determinísticos no Torch (Backend)
    if torch.cuda.is_available():
        #torch.cuda.manual_seed_all(seed) # Define a semente para todas as GPUs disponíveis (seed_everything já faz isso internamente).
        torch.use_deterministic_algorithms(True, warn_only=True) # Obriga o PyTorch a usar apenas algoritmos matemáticos que garantem o mesmo resultado bit a bit.
        
        # O CuDNN tem um recurso de "auto-tune". Na primeira iteração, ele testa vários algoritmos de convolução para ver qual é mais rápido para o hardware.
        # O algoritmo "vencedor" pode mudar dependendo da carga da máquina no momento. 
        # Desligar isso força o uso de um algoritmo padrão, garantindo consistência (à custa de performance).
        torch.backends.cudnn.benchmark = False  
        torch.backends.cudnn.deterministic = True # Reforça a obrigatoriedade de algoritmos determinísticos no CuDNN.
        
    print("✅ Ambiente configurado e blindado contra aleatoriedade.")

# =============================================================================
# 4. EXECUÇÃO IMEDIATA
# =============================================================================
set_reproducibility(global_seed)

🔒 Variáveis de ambiente de determinismo configuradas pré-importação.


Seed set to 42


🔒 Aplicando configurações de reprodutibilidade total (Seed: 42)...
✅ Ambiente configurado e blindado contra aleatoriedade.


In [2]:
# Classe para facilitar o uso de cores no terminal
class CoresTerminal:
    """Contém códigos ANSI para colorir o output no terminal."""
    VERMELHO = '\033[91m'
    VERDE = '\033[92m'
    FIM = '\033[0m'

In [3]:
# Funçao para avaliar previsões
def evaluate_simple_forecast(
    model,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    model_name: str = 'LSTM',  
    split: str ="",
    inteiro: bool = False,
    transf_log: bool = False # Alterado para transf_log com padrão False
) -> pd.DataFrame:
    """
    Executa avaliação de previsão considerando o nome do modelo dinâmico.
    Faz a previsão sem o uso de janela deslizande (rolling forecast).
    """
    print(f"\n{split} Iniciando a previsão com o modelo: {model_name}...")

    # --- 1. Preparação dos Dados ---
    history_df = train_df.copy()
    future_data_to_evaluate = test_df.copy()

    # Conversão de datas
    history_df['ds'] = pd.to_datetime(history_df['ds'])
    future_data_to_evaluate['ds'] = pd.to_datetime(future_data_to_evaluate['ds'])

    # Filtra histórico para evitar vazamento de dados
    cutoff = future_data_to_evaluate['ds'].min()
    history_df = history_df[history_df['ds'] < cutoff]
  
    # Prepara df futuro para predict (sem y)
    futr_df = future_data_to_evaluate.drop(columns=["y"], errors='ignore')

    # --- 2. Geração da Previsão ---
    forecasts_df = model.predict(
        df=history_df,
        futr_df=futr_df
    )
    
    # Verificação de segurança: A coluna do modelo existe?
    if model_name not in forecasts_df.columns:
        # Tenta fallback inteligente ou erro
        cols_disponiveis = [c for c in forecasts_df.columns if c not in ['unique_id', 'ds']]
        raise ValueError(f"A coluna '{model_name}' não foi encontrada na previsão. Colunas disponíveis: {cols_disponiveis}")

    print("Previsão concluída. Combinando com dados reais...")

    # --- 3. Pós-processamento e Avaliação ---
    evaluation_df = forecasts_df.merge(
        future_data_to_evaluate[["unique_id", "ds", "y"]],
        on=["unique_id", "ds"],
        how="inner"
    )

    # Tratamento de Log (Reversão)
    # Só executa se transf_log for True
    if transf_log:
        evaluation_df['y'] = np.expm1(evaluation_df['y'])
        evaluation_df[model_name] = np.expm1(evaluation_df[model_name]) # Usa model_name

    # Arredondamento
    if inteiro:
        evaluation_df[model_name] = evaluation_df[model_name].round().astype(int)
        evaluation_df['y'] = evaluation_df['y'].round().astype(int)

    # Cálculo da Diferença Percentual (passando o nome da coluna)
    evaluation_df = calcular_diferenca_percentual(evaluation_df, col_pred=model_name)

    print("Avaliação finalizada.")
    return evaluation_df


def calcular_diferenca_percentual(df: pd.DataFrame, col_pred: str = 'LSTM', col_real: str = 'y') -> pd.DataFrame:
    """
    Calcula a diferença percentual absoluta entre previsão e valor real.
    Retorna NaN para casos de divisão por zero ou dados ausentes (período de treino).
    """
    df = df.copy()
    
    # 1. Garante que as colunas sejam numéricas
    # 'errors="coerce"' é o segredo: ele transforma "-" e textos em NaN automaticamente
    pred_numeric = pd.to_numeric(df[col_pred], errors='coerce')
    real_numeric = pd.to_numeric(df[col_real], errors='coerce')
    
    # 2. Cálculo Vetorizado: |(Pred - Real) / Real| * 100
    # O Pandas/Numpy lida com NaNs automaticamente (NaN em qualquer operação resulta em NaN)
    diff_pct = ((pred_numeric - real_numeric) / real_numeric).abs() * 100
    
    # 3. Tratamento de Divisão por Zero
    # O cálculo acima gera 'inf' (infinito) se o real for 0.
    # Aqui substituímos 'inf' por NaN para manter a consistência.
    df['diferença_%'] = diff_pct.replace([np.inf, -np.inf], np.nan)
    
    # 4. Arredondamento (opcional, para limpeza visual)
    df['diferença_%'] = df['diferença_%'].round(2)
    
    return df

In [4]:
# Funão que testa o modelo salvo e salva resultados em CSV
def teste_modelo(
    local,              # Path onde o modelo treinado está salvo
    csv_dir,            # Path do arquivo CSV onde os resultados serão acumulados
    treino_id,          # ID único do experimento/treino
    dataset,            # DataFrame completo (Treino + Validação + Teste)
    train_ds,           # DataFrame apenas de Treino (usado para metadados)
    val_ds,             # DataFrame de Validação (pode ser None)
    test_ds,            # DataFrame de Teste
    comentario,         # String com observações do analista
    nome_dataset,       # Nome do dataset (ex: 'V43')
    hiperparametros,    # Dicionário com configs do modelo
    incluir_treino=False, # Flag pesada: se True, faz previsão no passado (In-Sample)
    transf_log=False      # Flag para indicar se os dados estão em Log
):
    """
    Carrega um modelo NeuralForecast salvo, gera previsões para Teste, Validação 
    e (opcionalmente) Treino, calcula métricas de erro e salva tudo em um CSV histórico.
    """

    print(f"\n>>> Iniciando Teste do Modelo carregado de: {local}")
    
    # Carrega o modelo pré-treinado
    model = NeuralForecast.load(path=f"{local}")
    
    # Define colunas padrão para garantir consistência
    colunas_finais = ['treino_id', 'unique_id', 'ds', 'y', 'y_pred', 'diferença_%', 'flag', 'dataset', 'modelo', 'comentario', 'data_treino']

    # -------------------------------------------------------------------------
    # FUNÇÃO AUXILIAR INTERNA
    # -------------------------------------------------------------------------
    def processar_dataframe(df_raw, flag_name):
        """
        Padroniza o dataframe de previsões: renomeia colunas, calcula erro e ajusta datas.
        """
        if df_raw is None or df_raw.empty:
            return None
        
        df_proc = df_raw.copy()
        
        # Identifica dinamicamente a coluna de previsão (que não seja unique_id, ds ou y)
        cols_reservadas = ['unique_id', 'ds', 'y', 'cutoff']
        candidatos_pred = [c for c in df_proc.columns if c not in cols_reservadas]
        
        # Se houver colunas de previsão, pega a primeira
        col_pred_name = candidatos_pred[0] if candidatos_pred else 'LSTM'
        
        # Padroniza nomes
        df_proc = df_proc.rename(columns={col_pred_name: 'y_pred'})
        
        # Garante apenas as colunas essenciais
        df_proc = df_proc[['unique_id', 'ds', 'y', 'y_pred']].copy()
        
        # Calcula Erro da diferença percentual
        df_proc = calcular_diferenca_percentual(df_proc, col_pred='y_pred', col_real='y')
        
        # Metadados
        df_proc['flag'] = flag_name
        
        # Padronização de Data para String (ISO format)
        df_proc['ds'] = (pd.to_datetime(df_proc['ds'], errors='coerce')
                         .fillna(pd.Timestamp.now())
                         .dt.strftime('%Y-%m-%dT%H:%M:%S'))
        
        return df_proc

    # -------------------------------------------------------------------------
    # 1. PREVISÕES NO CONJUNTO DE TESTE (Obrigatório)
    # -------------------------------------------------------------------------
    print("... Gerando previsões de Teste")
    # Agora passamos transf_log para a função de avaliação
    preds_test = evaluate_simple_forecast(
        model=model,
        train_df=dataset,
        test_df=test_ds,
        split="Teste",
        transf_log=transf_log 
    )
    dados_teste = processar_dataframe(preds_test, 'teste')

    # -------------------------------------------------------------------------
    # 2. PREVISÕES NO CONJUNTO DE VALIDAÇÃO (Se val_ds for diferente de None)
    # -------------------------------------------------------------------------
    dados_validacao = None
    if val_ds is not None:
        print("... Gerando previsões de Validação")
        preds_val = evaluate_simple_forecast(
            model=model,
            train_df=dataset,
            test_df=val_ds,
            split="Validação",
            transf_log=transf_log
        )
        dados_validacao = processar_dataframe(preds_val, 'validacao')

    # -------------------------------------------------------------------------
    # 3. PREVISÕES NO CONJUNTO DE TREINO (In-Sample)
    # -------------------------------------------------------------------------
    # Prepara o dataframe base de treino
    dados_treino = dataset[['unique_id', 'ds', 'y']].copy()
    
    # Reversão Log1p APENAS se transf_log for True
    if transf_log:
        dados_treino['y'] = np.expm1(dados_treino['y']) 
    
    # Define data de corte para separar o que é treino
    if val_ds is not None:
        data_inicio_corte = val_ds['ds'].min()
    else:
        data_inicio_corte = dados_teste['ds'].min()
        
    # Filtra apenas datas anteriores ao corte
    dados_treino = dados_treino[pd.to_datetime(dados_treino['ds']) < pd.to_datetime(data_inicio_corte)].copy()
    
    if incluir_treino:
        print("... Iniciando 'predict_insample' (Isso pode demorar!)")
        # Gera previsões dentro da amostra de treino
        insample_df = model.predict_insample(step_size=hiperparametros['h'])
        
        # Reversão da transformação logarítmica nas previsões e valores reais
        if transf_log:
            insample_df['y'] = np.expm1(insample_df['y'])
        
        # Encontra colunas de previsão no insample
        cols_pred_insample = [c for c in insample_df.columns if c not in ['unique_id', 'ds', 'cutoff', 'y']]
        col_model = cols_pred_insample[0] if cols_pred_insample else 'LSTM'
        
        if transf_log:
            insample_df[col_model] = np.expm1(insample_df[col_model])

        # Remove o período de "aquecimento" (context window)
        contexto = hiperparametros.get('input_size', 0)
        min_cutoff_valido = pd.to_datetime(insample_df['cutoff'].min()) + pd.DateOffset(years=contexto)
        insample_df = insample_df[insample_df['cutoff'] >= min_cutoff_valido].copy()

        # Merge para trazer a previsão para o dataframe de treino original
        insample_preds = insample_df[['unique_id', 'ds', col_model]].rename(columns={col_model: 'y_pred'})
        
        # Garante tipos de dados compatíveis para o merge
        dados_treino['ds'] = pd.to_datetime(dados_treino['ds'])
        insample_preds['ds'] = pd.to_datetime(insample_preds['ds'])
        
        dados_treino = dados_treino.merge(insample_preds, on=['unique_id', 'ds'], how='left')
        
        # Processa erro e formatação
        dados_treino = calcular_diferenca_percentual(dados_treino, col_pred='y_pred', col_real='y')
        dados_treino['flag'] = 'treino'
        dados_treino['ds'] = dados_treino['ds'].dt.strftime('%Y-%m-%dT%H:%M:%S')
        
    else:
        # Se não incluir treino, cria DF vazio estruturado
        dados_treino = pd.DataFrame(columns=['unique_id', 'ds', 'y', 'y_pred', 'diferença_%', 'flag'])

    # -------------------------------------------------------------------------
    # 4. CONSOLIDAÇÃO E METADADOS
    # -------------------------------------------------------------------------
    print("... Consolidando dados")
    
    # Lista de dataframes válidos
    dfs_para_concatenar = [df for df in [dados_treino, dados_validacao, dados_teste] if df is not None and not df.empty]
    dados_completos = pd.concat(dfs_para_concatenar, ignore_index=True)

    # Adiciona Metadados Globais
    dados_completos['treino_id'] = treino_id
    dados_completos['dataset'] = nome_dataset
    dados_completos['modelo'] = "LSTM"
    dados_completos['data_treino'] = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')

    # Gera o texto do comentário detalhado
    def get_period_text(df_periodo):
        if df_periodo is None or df_periodo.empty: return "sem dados"
        min_year = df_periodo['ds'].dt.year.min()
        max_year = df_periodo['ds'].dt.year.max()
        return f"{min_year}" if min_year == max_year else f"{min_year} a {max_year}"

    txt_val = get_period_text(val_ds)
    txt_test = get_period_text(test_ds)
    txt_train = get_period_text(train_ds)
    
    # Adicionando info sobre Log no comentário para rastreabilidade
    log_info = "Log Transform: Sim" if transf_log else "Log Transform: Não"

    full_comment = (
        f"{local}\n"
        f"Modelo LSTM. Treino: {txt_train}. Validação: {txt_val}. Teste: {txt_test}.\n"
        f"Dataset: {nome_dataset}. {log_info}.\n"
        f"Obs: {comentario}\n\n"
        f"Hyperparams:\n"
        f"input_size: {hiperparametros.get('input_size')}, "
        f"h: {hiperparametros.get('h')}, "
        f"lr: {hiperparametros.get('learning_rate')}, "
        f"batch: {hiperparametros.get('batch_size')}, "
        f"encoder n layers: {hiperparametros.get('encoder_n_layers')}, "
        f"decoder layers: {hiperparametros.get('decoder_layers')}"
    )
    dados_completos['comentario'] = full_comment

    # Ordenação final das colunas
    for col in colunas_finais:
        if col not in dados_completos.columns:
            dados_completos[col] = np.nan
            
    dados_completos = dados_completos[colunas_finais]

    # -------------------------------------------------------------------------
    # 5. SALVAMENTO NO ARQUIVO CSV (APPEND)
    # -------------------------------------------------------------------------
    
    pasta = os.path.dirname(csv_dir)
    if pasta and not os.path.exists(pasta):
        os.makedirs(pasta)

    if not os.path.exists(csv_dir):
        print(f"Arquivo {csv_dir} não existe. Criando novo arquivo.")
        df_final = dados_completos
    else:
        print(f"Adicionando resultados ao arquivo existente: {csv_dir}")
        df_antigo = pd.read_csv(csv_dir)
        df_final = pd.concat([df_antigo, dados_completos], ignore_index=True)

    df_final.to_csv(csv_dir, index=False)
    
    clear_output()
    print(f"✅ Sucesso! Previsões e métricas salvas em: {csv_dir}")

In [5]:
# Função para ler hiperparâmetros de um arquivo YAML do PyTorch Lightning

def get_hyperparameters_from_yaml(yaml_path: str) -> Optional[Dict[str, Any]]:
    """
    Lê um arquivo hparams.yaml gerado pelo PyTorch Lightning e extrai hiperparâmetros.
    
    Args:
        yaml_path (str): Caminho para o arquivo .yaml.

    Returns:
        dict: Dicionário com os parâmetros. Retorna valores padrão (None) para chaves ausentes.
        None: Se o arquivo não existir ou estiver corrompido.
    """
    
    # 1. Verificação de existência do arquivo antes de tentar abrir
    if not os.path.exists(yaml_path):
        print(f"ERRO: Arquivo não encontrado: {yaml_path}")
        return None

    # 2. Leitura do YAML
    try:
        with open(yaml_path, 'r', encoding='utf-8') as f:
            # UnsafeLoader é necessário pois o PL salva tags de objetos Python (!!python/object...)
            config_data = yaml.load(f, Loader=yaml.UnsafeLoader)
            
        if not config_data:
            print(f"AVISO: O arquivo {yaml_path} está vazio.")
            return None
            
    except yaml.YAMLError as e:
        print(f"ERRO: YAML corrompido ou inválido: {e}")
        return None
    except Exception as e:
        print(f"ERRO: Falha inesperada ao ler {yaml_path}: {e}")
        return None

    # Tratamento especial para dicionários aninhados (optimizer_kwargs)
    opt_kwargs = config_data.get('optimizer_kwargs') or {} # Garante que seja dict se for None
    
    hyperparameters = {
        # Parâmetros Estruturais
        'encoder_n_layers': config_data.get('encoder_n_layers'),
        'encoder_hidden_size': config_data.get('encoder_hidden_size'),
        'decoder_layers': config_data.get('decoder_layers'),
        'decoder_hidden_size': config_data.get('decoder_hidden_size'),
        'input_size': config_data.get('input_size'),
        
        # Parâmetros de Treino
        'learning_rate': config_data.get('learning_rate'),
        'batch_size': config_data.get('batch_size'),
        'steps': config_data.get('max_steps'),
        
        # Mapeamentos com renomeação
        'dropout': config_data.get('encoder_dropout'), # Renomeia encoder_dropout -> dropout
        'weight_decay': opt_kwargs.get('weight_decay', 0.0), # Pega de dentro do kwargs ou retorna 0
        
        # IMPORTANTE: O script de teste usa 'h' (horizonte).
        # Tenta pegar 'h' ou 'horizon'. Se não achar, tenta inferir ou deixa None.
        'h': config_data.get('h', config_data.get('horizon')) 
    }

    return hyperparameters

In [6]:
# Função para calcular a Média Simples do WMAPE por município

def calcular_media_wmape_simples(
    df: pd.DataFrame, 
    ponderar: bool = False, 
    caminho_area: str = "../Dataset/Area_colhida_V2.csv",
    pausar_alarme: bool = True  
) -> float:
    """
    Calcula o WMAPE com verificações de robustez.
    
    Modo 1 (ponderar=False): Média Simples dos WMAPEs por município.
        - Trata divisão por zero: se y=0 e y_pred!=0, retorna NaN (ignora o registro).
        - Se y=0 e y_pred=0 (acerto perfeito), retorna 0.0.
        
    Modo 2 (ponderar=True): WMAPE Global ponderado pela Área.
        - Verifica integridade do merge entre Previsão e Área.
        - Alerta em vermelho se houver perda de dados.
        - Se houver alerta e pausar_alarme=True, aguarda 20s antes de continuar.
    
    Args:
        df (pd.DataFrame): DataFrame com colunas 'unique_id', 'ds', 'y', 'LSTM'.
        ponderar (bool): Se True, calcula erro de volume (y * area).
        caminho_area (str): Caminho para o CSV de áreas.
        pausar_alarme (bool): Se True, pausa por 20s ao detectar erro de integridade (default: True).
        
    Returns:
        float: O valor do erro.
    """
    
    # Previne alterações no dataframe original
    df_calc = df.copy()

    # =========================================================================
    # MODO 1: Média Simples de Produtividade (Equidade Territorial)
    # =========================================================================
    if not ponderar:
        def _calcular_wmape_individual(grupo):
            valores_reais = grupo['y'].values
            valores_previstos = grupo['LSTM'].values
            
            soma_reais_abs = np.sum(np.abs(valores_reais))
            soma_erros_abs = np.sum(np.abs(valores_previstos - valores_reais))
            
            # PROTEÇÃO CONTRA DIVISÃO POR ZERO
            if soma_reais_abs < 1e-6: # Considera 0 (trata flutuação de float)
                if soma_erros_abs < 1e-6:
                    return 0.0 # Acertou que é 0 (erro 0)
                else:
                    return np.nan # Erro infinito. Retorna NaN para não sujar a média final.
                
            return soma_erros_abs / soma_reais_abs

        # Aplica cálculo por município
        wmapes_por_municipio = df_calc.groupby('unique_id')[['y', 'LSTM']].apply(_calcular_wmape_individual)
        
        # Calcula a média ignorando NaNs
        media_final = np.nanmean(wmapes_por_municipio)
        
        # Se todos forem NaN (caso extremo), retorna infinito
        if np.isnan(media_final):
            return float('inf')
            
        return media_final

    # =========================================================================
    # MODO 2: Cálculo Ponderado por Área (Equidade Econômica)
    # =========================================================================
    else:
        # 1. Carregar dados de Área
        try:
            df_area = pd.read_csv(caminho_area)
        except FileNotFoundError:
            raise FileNotFoundError(f"Arquivo de área não encontrado em: {caminho_area}")
            
        # 2. Padronização de Datas
        if not pd.api.types.is_datetime64_any_dtype(df_calc['ds']):
            df_calc['ds'] = pd.to_datetime(df_calc['ds'])
        
        df_calc['Ano_Merge'] = df_calc['ds'].dt.year
        
        # Contagem pré-merge para verificação
        n_linhas_antes = len(df_calc)
        ids_antes = set(df_calc['unique_id'].unique())
        
        # 3. Realizar o Merge (Inner Join)
        df_merged = df_calc.merge(
            df_area, 
            left_on=['unique_id', 'Ano_Merge'], 
            right_on=['Municipio', 'Ano'], 
            how='inner' 
        )
        
        # 4. VERIFICAÇÃO DE PERDA DE DADOS (CRÍTICO)
        n_linhas_depois = len(df_merged)
        
        if n_linhas_antes != n_linhas_depois:
            diff = n_linhas_antes - n_linhas_depois
            ids_depois = set(df_merged['unique_id'].unique())
            ids_perdidos = len(ids_antes - ids_depois)
            
            print(f"{CoresTerminal.VERMELHO}ALERTA DE INTEGRIDADE (WMAPE PONDERADO):")
            print(f"O merge com o arquivo de áreas resultou na perda de dados.")
            print(f"Linhas antes: {n_linhas_antes} | Linhas depois: {n_linhas_depois}")
            print(f"Registros perdidos: {diff}")
            print(f"Municípios totalmente excluídos do cálculo: {ids_perdidos}{CoresTerminal.FIM}")
            
            # Lógica de Pausa solicitada
            if pausar_alarme:
                print(f"{CoresTerminal.VERMELHO}Pausando por 20 segundos para leitura do erro...{CoresTerminal.FIM}")
                time.sleep(20)
            
            if n_linhas_depois == 0:
                print(f"{CoresTerminal.VERMELHO}ERRO CRÍTICO: Sobraram 0 linhas após cruzar com dados de área.{CoresTerminal.FIM}")
                return float('inf') 
        
        # 5. Cálculo do Erro Ponderado
        # Produtividade * Area = Volume Total
        y_vol_real = df_merged['y'] * df_merged['Area']
        y_vol_pred = df_merged['LSTM'] * df_merged['Area']
        
        numerador = np.sum(np.abs(y_vol_pred - y_vol_real))
        denominador = np.sum(np.abs(y_vol_real))
        
        # Proteção final contra denominador 0 global
        if denominador < 1e-6:
            return 0.0 if numerador < 1e-6 else float('inf')
            
        return numerador / denominador

In [7]:
# Função para filtrar datasets por integridade dos dados (manter apenas municípios todos os registros no período de validação)
from typing import Tuple
def filtrar_datasets_por_integridade(
    train_df: pd.DataFrame, 
    val_df: pd.DataFrame, 
    test_df: pd.DataFrame
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Filtra os datasets de treino, validação e teste para manter apenas os
    municípios ('unique_id') que possuem dados completos no período de validação.
    Ou seja, se validação tem 12 meses, mantém apenas os municípios que têm os 12 meses completos.

    Args:
        train_df (pd.DataFrame): DataFrame de treino.
        val_df (pd.DataFrame): DataFrame de validação. Usado como referência para a verificação.
        test_df (pd.DataFrame): DataFrame de teste.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]: Uma tupla contendo os 
        DataFrames de treino, validação e teste devidamente filtrados.
    """
    print("--- Iniciando verificação de integridade dos dados ---")
    
    # Usa o DataFrame de validação como referência para a integridade.
    # 1. Conta o número de timestamps únicos que cada município DEVERIA ter.
    timestamps_esperados = len(val_df['ds'].unique())
    if timestamps_esperados == 0:
        print(f"{CoresTerminal.VERMELHO}ALERTA: O conjunto de validação está vazio. Nenhum filtro será aplicado.{CoresTerminal.FIM}")
        return train_df, val_df, test_df

    # 2. Conta quantos timestamps únicos cada município realmente POSSUI.
    contagem_por_id = val_df.groupby('unique_id')['ds'].nunique()
    
    # 3. Identifica os municípios com dados completos.
    ids_completos = contagem_por_id[contagem_por_id == timestamps_esperados].index
    
    # 4. Compara com o total de municípios para ver se a filtragem é necessária.
    ids_originais = val_df['unique_id'].nunique()
    
    if len(ids_completos) < ids_originais:
        total_removido = ids_originais - len(ids_completos)
        
        print(f"{CoresTerminal.VERMELHO}"
              "----------------------------------------------------------------------\n"
              "ALERTA: Inconsistência de dados encontrada.\n"
              f"Foram encontrados {total_removido} de {ids_originais} municípios com dados INCOMPLETOS no período de validação.\n"
              "Todos os datasets serão filtrados para manter apenas os municípios com dados completos.\n"
              f"----------------------------------------------------------------------{CoresTerminal.FIM}")
        
        # Filtra TODOS os dataframes para manter apenas os municípios com dados completos.
        # O uso de .copy() evita o SettingWithCopyWarning do pandas.
        train_filtrado = train_df[train_df['unique_id'].isin(ids_completos)].copy()
        val_filtrado = val_df[val_df['unique_id'].isin(ids_completos)].copy()
        test_filtrado = test_df[test_df['unique_id'].isin(ids_completos)].copy()
        
        # Se após a filtragem não sobrar nenhum dado, interrompe a execução.
        if val_filtrado.empty:
            print(f"{CoresTerminal.VERMELHO}ALERTA: Após a filtragem, não restaram dados válidos. Retornando DataFrames vazios.{CoresTerminal.FIM}")
            return train_filtrado, val_filtrado, test_filtrado
        
        print(f"Filtragem concluída. {len(ids_completos)} municípios mantidos.")
        return train_filtrado, val_filtrado, test_filtrado
        
    else:
        print(f"{CoresTerminal.VERDE}Verificação concluída. Todos os {ids_originais} municípios possuem dados completos no período de validação.{CoresTerminal.FIM}")
        return train_df, val_df, test_df

In [8]:
# Função para carregar e processar o dataset
def get_dataset(dataset_file, transf_log=False):
    """
    Carrega, processa e limpa o dataset, garantindo que todas as séries temporais
    para cada 'unique_id' estejam completas no intervalo de datas do dataset.
    
    Args:
        dataset_file (str): Caminho para o arquivo CSV.
        transf_log (bool): Se True, aplica transformação logarítmica (np.log1p) 
                           em colunas específicas. Padrão é False.
    """
    dataset = pd.read_csv(dataset_file)
    
    # --- Aplica transformação logarítmica (OPCIONAL) ---
    if transf_log:
        log_cols = [
            'Área colhida (Hectares)',
            'target',
            'precomediocafe', # a partir de novembro 2025 (V37)
            'Área destinada à colheita (Hectares)'
        ]
        existing_log_cols = [col for col in log_cols if col in dataset.columns]
        if existing_log_cols:
            for col in existing_log_cols:
                print(f"Aplicando transformação logarítmica na coluna: {col}")
                
            dataset[existing_log_cols] = dataset[existing_log_cols].apply(lambda x: np.log1p(x))

    # --- Renomeação e formatação para o padrão NeuralForecast ---
    dataset = dataset.rename(columns={
        "municipio": "unique_id",
        "ano": "ds",
        "target": "y"
    })
    
    # Ordenar por unique_id e ds (ano)
    dataset = dataset.sort_values(by=["unique_id", "ds"]).reset_index(drop=True)
    dataset['ds'] = pd.to_datetime(dataset['ds'].astype(str) + '-12-31')
    
    return dataset

# Treinamento

In [9]:
json_filename = "Teste_1_reuniao_16_dez_Ajuste_parametros_por_cluster_modelo_simetrico_sem_transomacao_log_WMAPE_ponderado_completo/Teste_1_resultados_otimizacao_geral_16_01_2026.json"
dataset_path  = "../Dataset/V44"


comentario = f"""21_01_2026: Ajuste de hiperparâmetros com Optuna.
Dataset V44 com 5 clusters.
Treinado para os anos de 2019-2024 para validação.
Neste não sera usada a transformação logarítmica.
Neste teste a seed do modelo sera ajustada, sendo uma seed por modelo.
Seeds usadas:
   - global seed: {global_seed} # Seed para os pontos que devem ser definidos antes de iniciar algumas bibliotecas."""

In [10]:
# Lista de parametros usados no modelo.
# Os parametros com valor -42 serao sobrescritos pelo valor no json (json_filename).
hiperparametros = { # Não sera usado pois esta tudos salvo no json
    'h': 1, # Valor real
    'input_size': -1, # Use -1 para janela de contexto máxima (pode ser sobrescrito pelo valor do json)
    'batch_size': -42, # -42 é so um valor para saber que sera sobrescrito (se tiver -42 nos dados final deve ter algum erro)
    'dropout': -42,
    'encoder_n_layers': -42,
    'learning_rate': -42, 
    'encoder_hidden_size': -42,
    'decoder_layers': -42,
    'decoder_hidden_size': -42,
    'weight_decay': -42,
    'steps': -42
}

# Para que o script pare exatamente no número total desejado ao reiniciar, é necessário calcular quantos trials restam antes de executar novamente.
# Ou seja, se o objetivo é 300 trials e já foram feitos 290, deve-se colocar N_TRIALS = 10
N_TRIALS = 150  # Número total de trials desejados (ajuste conforme necessário)
transf_log = False  # Defina como True se quiser aplicar transformação logarítmica
ponderar_erro = True  # Defina como True para calcular WMAPE ponderado por área

In [11]:
def objective(trial):
    # --- CÁLCULO DO TEMPO ESTIMADO ---
    elapsed_time = time.time() - study_start_time
    completed_trials = trial.number 
    
    if completed_trials > 0:
        avg_time_per_trial = elapsed_time / completed_trials
        remaining_trials = N_TRIALS - completed_trials
        eta_seconds = avg_time_per_trial * remaining_trials
        
        tempo_decorrido_str = str(timedelta(seconds=int(elapsed_time)))
        eta_str = str(timedelta(seconds=int(eta_seconds)))
    else:
        tempo_decorrido_str = "0:00:00"
        eta_str = "Calculando..."
        
    # 1. Sugestão de Hiperparâmetro (Seed)
    seed_trial = trial.suggest_int("random_seed", 1, 100000)

    # --- Configuração de Pastas ---
    base_folder = os.getcwd()
    
    # Pasta específica para salvar os CSVs desta seed
    csv_output_folder = os.path.join(base_folder, "csv_models", str(seed_trial))
    os.makedirs(csv_output_folder, exist_ok=True)

    # Pasta temporária para o modelo (será apagada a cada iteração)
    temp_model_path = os.path.join(base_folder, "temp_models", f"trial_{trial.number}")
    
    clear_output(wait=True) 
    print(f"==================================================")
    print(f" TRIAL ATUAL: {trial.number + 1}/{N_TRIALS}")
    print(f" Seed: {seed_trial}")
    print(f" Log Transform: {transf_log}") # Info visual
    print(f" Wmape ponderado: {ponderar_erro}") # Info visual
    print(f" Tempo Decorrido: {tempo_decorrido_str}")
    print(f" Tempo Restante (ETA): {eta_str}")
    print(f" Processando: {dataset_file} | Val: {ano_val}")
    print(f"==================================================\n")

    set_reproducibility(seed_trial)
        
    model = LSTM(
        h=params_run['h'],
        input_size= params_run['input_size'],
        batch_size=params_run['batch_size'],
        scaler_type="revin",
        encoder_dropout=params_run['dropout'],
        encoder_n_layers=params_run['encoder_n_layers'],
        encoder_hidden_size=params_run['encoder_hidden_size'],
        decoder_layers=params_run['decoder_layers'],
        decoder_hidden_size=params_run['decoder_hidden_size'],
        futr_exog_list=exog_list,
        learning_rate=params_run['learning_rate'],
        max_steps=params_run['steps'],
        loss=HuberLoss(delta=1.0),
        optimizer=AdamW,
        optimizer_kwargs={"weight_decay": params_run['weight_decay']},
        lr_scheduler=StepLR,
        lr_scheduler_kwargs={"step_size": int(params_run['steps'] * 0.5), "gamma": 0.1},
        random_seed=seed_trial
    )

    nf = NeuralForecast(models=[model], freq="YE")
    nf.fit(df=train_ds, verbose=False)

    # --- Previsão (Para métrica do Optuna) ---
    combined_df = evaluate_simple_forecast(
        model=nf,
        train_df=train_ds,
        test_df=val_ds,
        transf_log=transf_log
    )
        
    '''
    # --- Salvamento e Teste do Modelo ---
    # 1. Salva o modelo temporariamente (overwrite garante que não acumule lixo se a pasta já existir)
    nf.save(path=temp_model_path, overwrite=True)
    
    
    # 2. Define o caminho final do CSV conforme solicitado
    nome_dataset_limpo = dataset_file[4:-4] # Mantendo sua lógica de slice
    csv_filename = f"{ano_val}_{nome_dataset_limpo}.csv"
    csv_full_path = os.path.join(csv_output_folder, csv_filename)
    
    
    # 3. Executa a função de teste que gera o CSV
    teste_modelo(
        local = temp_model_path,              # Usa a pasta temporária
        csv_dir = csv_full_path,              # Salva na pasta organizada por seed
        treino_id = f"{ano_val}_{nome_dataset_limpo}",
        dataset = dataset,
        train_ds = train_ds,
        val_ds = val_ds,
        test_ds = test_ds,
        comentario = "ajuste de seed",
        nome_dataset = "V44",
        hiperparametros = params_run,
        incluir_treino = False,
        transf_log = transf_log 
    )
    
    # 4. LIMPEZA IMEDIATA: Remove a pasta do modelo salvo (checkpoint)
    shutil.rmtree(temp_model_path, ignore_errors=True)

    # Limpeza de Memória RAM/VRAM
    del model
    del nf
    gc.collect()

    
    # --- Pruning (Otimização) ---
    # Caso o teste inicial mostre que o modelo está ruim, interrompe o trial cedo.
    erro_cluster_atual = calcular_media_wmape_simples(combined_df, ponderar=ponderar_erro)
    erros_parciais.append(erro_cluster_atual)
    media_erro_atual = np.mean(erros_parciais)
    
    trial.report(media_erro_atual, step=step_count)
    step_count += 1
    
    if trial.should_prune():
        # Limpa qualquer resíduo antes de sair
        shutil.rmtree(temp_model_path, ignore_errors=True)
        raise optuna.TrialPruned()
    '''
    
        
    score_final_wmape = calcular_media_wmape_simples(combined_df, ponderar=ponderar_erro)
    
    return score_final_wmape

In [12]:
# # Lê os dados do arquivo JSON, onde estão os resultados da otimização do Optuna anteriormente feita.
if 'json_filename' in locals() and os.path.exists(json_filename):
    with open(json_filename, 'r', encoding='utf-8') as arquivo:
        # Carrega o conteúdo do JSON para uma variável Python (dicionário)
        dados = json.load(arquivo)
        print(f"✅ Arquivo JSON carregado: {json_filename}")
else:
    # Fallback caso o snippet seja rodado isoladamente para teste
    dados = [] 
    print(f"{CoresTerminal.VERMELHO}AVISO: Arquivo JSON não encontrado. A lista de dados está vazia.{CoresTerminal.FIM}")

# Defina o nome do arquivo JSON onde todos os resultados serão acumulados
json_resultados = "otimizacao_seed_por_modelo_21_01_2026.json"

# Define um nome para o arquivo de banco de dados
storage_name = "sqlite:///optuna_study.db"

for dado in dados:
    study_start_time = time.time() # Marca o início do estudo para medição de tempo
    
    ano_val = dado['ano_val'] # 
    dataset_file = dado['dataset_file']
    
    # Define o ano de teste e carrega dataset
    ano_teste = ano_val + hiperparametros['h'] 
    dataset_path_ano = f"{dataset_path}/{ano_teste}"
    
    dataset = get_dataset(f"{dataset_path_ano}/{dataset_file}", transf_log=transf_log)   

    # --- Coleta a lista variaveis exogenas ---
    exog_list = [col for col in dataset.columns.tolist() if col not in ["ds", "y", "unique_id"]]

    # --- Preparação do dataset de treino, validação e teste ---
    train_ds = dataset[dataset['ds'].dt.year < ano_val].copy()
    val_ds = dataset[(dataset['ds'].dt.year >= ano_val) & (dataset['ds'].dt.year < ano_teste)].copy()
    test_ds = dataset[dataset['ds'].dt.year >= ano_teste].copy()

    # --- Aplica a função de filtragem ---
    train_ds, val_ds, test_ds = filtrar_datasets_por_integridade(
        train_df=train_ds,
        val_df=val_ds,
        test_df=test_ds
    )

    # -- Configura parametros do modelo ---
    params_run = hiperparametros.copy()
    params_run['encoder_hidden_size'] = dado['best_params']['hidden_size']
    params_run['decoder_hidden_size'] = dado['best_params']['hidden_size']
    params_run['learning_rate'] = dado['best_params']['learning_rate']
    params_run['encoder_n_layers'] = dado['best_params']['n_layers']
    params_run['weight_decay'] = dado['base_params']['weight_decay']
    params_run['decoder_layers'] = dado['best_params']['n_layers']
    params_run['input_size'] = dado['base_params']['input_size']
    params_run['batch_size'] = dado['base_params']['batch_size']
    params_run['steps'] = dado['base_params']['steps']
    params_run['h'] = dado['base_params']['h']    
    
    # Se n_layers for 1, dropout é 0, pois não faz sentido ter dropout em 1 layer.
    params_run['dropout'] = dado['best_params'].get('dropout', dado['base_params']['dropout'])

    if params_run['input_size'] == -1: # Se não for -1 ele usa o que esta no json.
        params_run['input_size'] = len(train_ds['ds'].unique()) - params_run['h']


    # ======= Optuna =======
    # Instancia o sampler (mantenha a seed para consistência na lógica de sugestão)
    sampler = TPESampler(seed=global_seed)

    # Crie (ou carregue) o estudo
    study = optuna.create_study(
        study_name=f"otimizacao_seed_{dataset_file[:-4]}_{ano_val}", # O nome deve ser o mesmo para retomar
        storage=storage_name,                 
        load_if_exists=True,                  
        direction="minimize",
        sampler=sampler
    )
    print(f"Estudo carregado com {len(study.trials)} trials já realizados.") 

    # ======= Inserir Trial Inicial =======
    params_iniciais = {
        "random_seed": 42,
    }
    study.enqueue_trial(params_iniciais)

    # 4. Executa a otimização
    study.optimize(objective, n_trials=N_TRIALS, n_jobs=1)
    df = study.trials_dataframe()
    df.to_csv(f"otimizacao_seed_por_cluster_{dataset_file[:-4]}_{ano_val}.csv", index=False)

    clear_output()
    print("=== Otimização concluída ===")
    print("Melhor valor WMAPE:", study.best_value)
    print("Melhores parâmetros:", study.best_params)

    # ======= SALVAMENTO NO JSON =======

    # Cria o dicionário com os dados deste loop
    resultado_atual = {
        "ano_val": ano_val,
        "dataset_file": dataset_file,
        "global_seed": global_seed,
        "best_value": study.best_value,
        "best_params": study.best_params, # seed do modelo
        "base_params": params_run,        # parametros fixos usados (estou repasando para não precisar usar arquivos antigos)
        "transf_log": transf_log,
        "wmape_ponderado_area": ponderar_erro,
        "arquivo_com_parametros_base": json_filename,
        "dataset_path": dataset_path,
        "comentario": comentario    
    }
    dados_acumulados = []

    # 1. Tenta ler o arquivo se ele existir
    if os.path.exists(json_resultados):
        try:
            with open(json_resultados, "r", encoding="utf-8") as f:
                dados_acumulados = json.load(f)
        except json.JSONDecodeError:
            # Se o arquivo estiver corrompido ou vazio, inicia lista vazia
            dados_acumulados = []

    # 2. Adiciona o novo resultado
    dados_acumulados.append(resultado_atual)

    # 3. Salva a lista atualizada de volta no arquivo
    with open(json_resultados, "w", encoding="utf-8") as f:
        json.dump(dados_acumulados, f, indent=4, ensure_ascii=False)
        
    print(f"Resultados salvos com sucesso em: {json_resultados}")

=== Otimização concluída ===
Melhor valor WMAPE: 0.08974040933951423
Melhores parâmetros: {'random_seed': 7052}
Resultados salvos com sucesso em: otimizacao_seed_por_modelo_21_01_2026.json
